In [ ]:
import numpy as np
## Note students there might be compatibility issues if you have curret version of numpy.
'''
      ********  CAUTION ***********
      1) Step 1:  Check what version of numpy you have installed. If it is any version above numpy-1.20 or higher, gymnasium or gym environment has removed
      some data types and Discrete boolean values conversion is not supported.
      2) Step 2: Either unistall the numpy package first and re-install a lower version. or uncomment the below lines.
         a) If you are uncommenting. Make sure it is loaded before gym otherwise you might get compatibility issues in wrong places which will be difficult to encounter.
'''
#if not hasattr(np, 'bool8'):
#    np.bool8 = np.bool_
import gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
#Don't chnage the number of layers in MLP. You can do that for your own testing but while submitting keep the structure same. However, change the number of neurons in each layer.
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
# Load dataset: offline buffer with transitions (s, a, r, s', done)

'''
    In this part load the two different initial policies asked in the question, one given as a pickle file and the other as the heuristic mentioned
'''
def generate_cartpole_dataset(env_name="CartPole-v1", episodes=100):
    env = gym.make(env_name)
    data = []

    def policy(obs):
        ##############################
              #Your code here
        ##############################

    for _ in range(episodes):
        obs = env.reset()
        done = False
        while not done:
            act = policy(obs)
            next_obs, reward, done, _ = env.step(act)
            data.append((obs, act, reward, next_obs, float(done)))
            obs = next_obs

    return data

In [ ]:
def train_iql(data, epochs=100, batch_size=64, discount=0.99, beta=3.0):
    obs_dim = len(data[0][0])
    n_actions = 2

    q_net = MLP(obs_dim, n_actions).to(device)
    target_q_net = MLP(obs_dim, n_actions).to(device)
    target_q_net.load_state_dict(q_net.state_dict())

    v_net = MLP(obs_dim, 1).to(device)

    q_opt = torch.optim.Adam(q_net.parameters(), lr=1e-3)
    v_opt = torch.optim.Adam(v_net.parameters(), lr=1e-3)

    plotting_data = {'epoch':[],'Q_loss':[],'V loss':[],'value_function':[]}

    for epoch in range(epochs):
      random.shuffle(data)
      ######################################################
                  #YOUR IMPLEMENTATION OF IQL here
      ######################################################
        plotting_data['epoch'].append(epoch)
        plotting_data['Q_loss'].append(q_loss.item())
        plotting_data['V loss'].append(v_loss.item())
        plotting_data['value_function'].append(vf)
        print(f"[IQL] Epoch {epoch}, Q Loss: {q_loss.item():.4f}, V Loss: {v_loss.item():.4f}")
    return plotting_data

In [ ]:
def train_cql(data, epochs=100, batch_size=64, discount=0.99, alpha=1.0):
    obs_dim = len(data[0][0])
    n_actions = 2

    q_net = MLP(obs_dim, n_actions).to(device)
    target_q_net = MLP(obs_dim, n_actions).to(device)
    target_q_net.load_state_dict(q_net.state_dict())
    optimizer = torch.optim.Adam(q_net.parameters(), lr=1e-3)
    plotting_data = {'epoch':[],'Q_loss':[],'CQL_penalty':[],'value_function':[]}

    for epoch in range(epochs):
        random.shuffle(data)
        ############################################################
                  #YOUR IMPLEMENTATION OF CQL here
        ############################################################
        plotting_data['epoch'].append(epoch)
        plotting_data['Q_loss'].append(loss.item())
        plotting_data['CQL_penalty'].append(cql_penalty.item())
        plotting_data['value_function'].append(vf)
        print(f"[CQL] Epoch {epoch}, Loss: {loss.item():.4f}, Penalty: {cql_penalty.item():.4f}")
    return plotting_data

In [ ]:
dataset = generate_cartpole_dataset()
train_iql(dataset, epochs=2)
train_cql(dataset, epochs=2)

<ipython-input-11-1ba6f759a33c>:20: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  obs = torch.tensor(obs, dtype=torch.float32).to(device)


[IQL] Epoch 0, Q Loss: 61.0594, V Loss: -41.7863
[IQL] Epoch 1, Q Loss: 3862.3115, V Loss: -2329.8450
[CQL] Epoch 0, Loss: 0.5670, Penalty: 0.5549
[CQL] Epoch 1, Loss: 0.5252, Penalty: 0.4264


In [ ]:
cql_data = train_cql(dataset, epochs=100)
iql_data = train_iql(dataset, epochs=100)

from matplotlib import pyplot as plt
plt.plot(cql_data['epoch'], cql_data['Q_loss'], label='CQL-loss')
plt.plot(iql_data['epoch'], iql_data['Q_loss'], label='IQL-loss')
plt.legend()
plt.title('Loss vs Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()

In [ ]:
plt.figure()
plt.plot(cql_data['epoch'], cql_data['CQL_penalty'], label='CQL-penalty')
plt.plot(iql_data['epoch'], iql_data['CQL_penalty'], label='IQL-penalty')
plt.legend()
plt.title('Penalty vs Epochs')
plt.xlabel('Epochs')
plt.ylabel('Penalty')
plt.show()

In [ ]:
plt.figure()
plt.plot(cql_data['epoch'], cql_data['value_function'], label='CQL-value')
plt.plot(iql_data['epoch'], iql_data['value_function'], label='IQL-value')
plt.legend()
plt.title('Value vs Epochs')
plt.xlabel('Epochs')
plt.ylabel('Value')
plt.show()